<font size=6>**Finetuning LLM**</font>

<font size=5>**Fine-Tuned AI for Summarizing Insurance Sales Conversations**</font>

This project focuses on fine-tuning a large language model (LLM) to generate structured, context-aware summaries of insurance sales conversations. The goal is to transform fragmented, unstructured communication (emails, call transcripts, CRM notes, meeting summaries) into concise, actionable briefings that support enterprise sales workflows.

+ The model is designed to extract and synthesize:

+ Client priorities and product interests

+ Objections and risk signals

+ Open action items and prior commitments

+ Renewal context and upsell/cross-sell opportunities

Rather than acting as a generic summarizer, the model is fine-tuned to reflect the language, structure, and decision-making needs specific to insurance sales environments.

<br>

#### **Business Challenge:**

In enterprise insurance sales, critical relationship context accumulates over time across multiple communication channels. However:

1. Communication data is largely unstructured and siloed across systems.

2. Manual review of historical conversations is time-intensive and inconsistent.

3. Important signals (churn risk, pricing concerns, competitive threats, expansion interest) are often buried in long threads or transcripts.

4. Preparation quality varies significantly between representatives, impacting renewal performance and client experience.

These inefficiencies create measurable business risk: missed upsell opportunities, weakened personalization, and avoidable client churn.

#### **Solution**
Provide a Custom Fine-Tuned AI Model for Sales Interaction Summarization

To address this challenge, we propose training a domain-specific fine-tuned language model tailored for enterprise insurance communication.
The model will:

1. Ingest few multi-modal inputs (emails, transcripts, notes).
2. Identify intent, extract key discussion points, client interests, pain points, and commitments.
3. Generate concise, actionable summaries under 200 words, customized for enterprise insurance workflows.
4. Be fine-tuned on real-world communication data to learn domain-specific vocabulary and interaction patterns.

This AI-powered tool will augment sales productivity, enhance client engagement, and ensure consistent follow-ups—turning scattered conversations into strategic intelligence.

# **Setup**

In [2]:

# Do this only in Colab notebooks! If using a local env then use cell below this one to ensure you have an NVDA GPU as required by unsloth.

%pip install bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
%pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
%pip install transformers==4.51.3
%pip install unsloth

%pip install -q datasets evaluate bert-score

INFO: pip is looking at multiple versions of unsloth-zoo to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth-zoo to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB

In [ ]:
# Check if GPU is available and print details

import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU count: 1
GPU name: Tesla T4


In [2]:
%pip show unsloth

Name: unsloth
Version: 2026.2.1
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: http://www.unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, bitsandbytes, datasets, diffusers, hf_transfer, huggingface_hub, numpy, packaging, peft, protobuf, psutil, sentencepiece, torch, torchvision, tqdm, transformers, triton, trl, tyro, unsloth_zoo, wheel, xformers
Required-by: 


In [ ]:
# Import necessary libraries

from unsloth import FastLanguageModel
import torch
import evaluate
from tqdm import tqdm
import pandas as pd
from datasets import Dataset

from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **1. Evaluation of LLM before FineTuning**

**Loading the Testing Data**


In [ ]:
testing_data = pd.read_csv("/content/finetuning_testing.csv")

test_dialogues = [sample for sample in testing_data['Dialogues']]
test_summaries = [sample for sample in testing_data['Summary']]

**Loading the Mistral Vanilla Model**

In [ ]:
# Load the model and tokenizer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
FastLanguageModel.for_inference(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layer

**Inference**


In [8]:
summarization_prompt_template = """[INST]
Summarize the dialogue mentioned in the user input. Be specific and concise in your summary.
Ensure that you retain the entities mentioned in the dialogue in your summary.

### User Input:
{dialogue}
[/INST]
"""

In [9]:
predicted_summaries = []

In [ ]:
# Generate summaries for the test dialogues

for gold_dialogue in tqdm(test_dialogues):

    try:
        prompt = summarization_prompt_template.format(dialogue=gold_dialogue)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=0,
            pad_token_id=tokenizer.eos_token_id
        )

        prediction = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[-1]:],
            skip_special_tokens=True,
            cleanup_tokenization_spaces=True
        )

        predicted_summaries.append(prediction)

    except Exception as e:
        print(e) # log error and continue
        continue

100%|██████████| 10/10 [02:28<00:00, 14.86s/it]


### Evaluation


In [11]:
predicted_summaries

["The user is inquiring about the feasibility of the sales representative's plans for covering multiple states in their expanded business. The sales representative confirms that they offer multi-state coverage with unified billing and compliance alignment. The user asks if regional variances affect plan design or premium structure, to which the sales representative responds that while premiums can vary based on state regulations and provider networks, the core benefits remain consistent. The user requests information on how compliance is managed across state lines, and the sales representative explains that they have a regulatory team in place to monitor each jurisdiction and update plans accordingly. Lastly,",
 'The user inquired about cyber insurance options to protect against data breaches and operational downtime due to a recent ransomware scare. The sales representative suggested cyber liability plans that cover expenses for forensic investigations, data restoration, and crisis PR

In [12]:
bert_scorer = evaluate.load("bertscore")

In [13]:
score = bert_scorer.compute(
    predictions=predicted_summaries,
    references=test_summaries,
    lang='en',
    rescale_with_baseline=True
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
sum(score['f1'])/len(score['f1'])

0.16142076402902603

**Sub-optimal result before Fine-Tuning**


# **2. FineTuning an LLM**

In [39]:
# TorchDynamo/compile acceleration may error depending on environment; disable with TORCHDYNAMO_DISABLE=1

import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

### Data Preparation

In [28]:
training = pd.read_csv("/content/finetuning_training.csv")
training_dict = training.to_dict(orient='list')

# Create a dataset from the dictionary
training_dataset = Dataset.from_dict(training_dict)

In [ ]:
# Define the end-of-sequence token for the model
EOS_TOKEN = tokenizer.eos_token

**The Alpaca instruction prompt is a general purpose prompt template that can be adapted to any task.**

In [30]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
# Define a function to format the prompt for each example in the dataset using the template and add the end-of-sequence token at the end of the prompt

def prompt_formatter(example, prompt_template):
    instruction='Write a concise summary of the following dialogue.'
    dialogue=example["Dialogues"]
    summary=example["Summary"]

    formatted_prompt = prompt_template.format(instruction, dialogue, summary) + EOS_TOKEN

    return {'text': formatted_prompt}

#  Add the end-of-sequence token to the prompt i.e. we're adding a special marker at the end of the prompt to show it's finished

In [ ]:
# Apply the prompt formatter to the training dataset to create a new dataset with formatted prompts for each example

formatted_training_dataset = training_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [33]:
validation=pd.read_csv("/content/finetuning_validation.csv")
validation_dict =validation.to_dict(orient='list')

# Create a dataset from the dictionary
validation_dataset = Dataset.from_dict(validation_dict)

In [ ]:
# Apply the prompt formatter to the training dataset to create a new dataset with formatted prompts for each example

formatted_validation_dataset = validation_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

**Fine-Tuning**

In [35]:
# patch in the adapter modules to the base model using the get_peft_model method.

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
    random_state=42,
    loftq_config=None
)

Unsloth: Already have LoRA adapters! We shall skip this step.


Practical Tip:  r  defines the dimensions of the low-rank matrices, while  α  determines the scaling factor for the weight matrices. It is common to freeze  α=16 , while varying the values of  r=α,α/2,α/4  and arriving at the optimal value of that gives the lowest validation loss (note that we use the same loss used for the base model, e.g., perplexity or log loss)

In [36]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [ ]:
# Define the SFTTrainer with the model, tokenizer, datasets, and training arguments. This will handle the fine-tuning process.

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_training_dataset,
    eval_dataset = formatted_validation_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
# Start the fine-tuning process. This will train the model on the formatted training dataset and evaluate on the validation dataset according to the specified training arguments.

training_history = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,2.388400
2,2.406600
3,2.202100
4,1.798800
5,1.481100
6,1.229600
7,1.177300
8,0.691300
9,0.292700
10,0.140300


**Saving the Trained Model**


In [ ]:
# Setup to enable bash commands

import locale

def getpreferredencoding():
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [ ]:
# We will be saving this fine tuned model locally so that we can test/evaluate this model. As finetuning is an expensive process, its best to save the model, in case of crashes.

import os, shutil

lora_model_name = "finetuned_mistral_lora"
local_dir = f"./artifacts/{project_name}"

# Always save locally
os.makedirs(local_dir, exist_ok=True)
model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)

print("Saved locally to:", local_dir)
!ls -lh "{local_dir}"

Saved locally to: ./artifacts/finetuned_Buttcheecks_mistral_lora
total 164M
-rw-r--r-- 1 root root 1.2K Feb 23 05:30 adapter_config.json
-rw-r--r-- 1 root root 161M Feb 23 05:30 adapter_model.safetensors
-rw-r--r-- 1 root root 1.1K Feb 23 05:30 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Feb 23 05:29 README.md
-rw-r--r-- 1 root root  552 Feb 23 05:30 special_tokens_map.json
-rw-r--r-- 1 root root 1.1K Feb 23 05:30 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Feb 23 05:30 tokenizer.json
-rw-r--r-- 1 root root 482K Feb 23 05:30 tokenizer.model


# **3. Evaluation of LLM after FineTuning**

**Loading the Testing Data**

In [56]:
testing_data = pd.read_csv("/content/finetuning_testing.csv")

test_dialogues = [sample for sample in testing_data['Dialogues']]
test_summaries = [sample for sample in testing_data['Summary']]

**Loading the Finetuned Mistral LLM**

In [70]:
!cp -r /content/artifacts/finetuned_mistral_lora /content/finetuned_mistral_lora

In [71]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


**Inferencing**

In [72]:
alpaca_prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a concise summary of the following dialogue.

### Input:
{}

### Response:
{}
"""

In [73]:
predicted_summaries = []

In [ ]:
# Generate summaries for the test dialogues using the fine-tuned model

for gold_dialogue in tqdm(test_dialogues):

    try:
        prompt = alpaca_prompt_template.format(gold_dialogue, '')
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=0,
            pad_token_id=tokenizer.eos_token_id
        )

        prediction = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[-1]:],
            skip_special_tokens=True,
            cleanup_tokenization_spaces=True
        )

        predicted_summaries.append(prediction)

    except Exception as e:
        print(e) # log error and continue
        continue

100%|██████████| 10/10 [00:29<00:00,  2.98s/it]


### Evaluation

In [75]:
predicted_summaries

['Client inquired about coverage across multiple states and its impact on premiums and compliance. Action: Share case file and compliance checklist.',
 'Client inquired about cyber insurance options for data breaches and operational downtime, its integration with current liability coverage, typical payout structure, and sample policy and incident response playbook. Action: Provide sample policy and incident response playbook by tomorrow morning.',
 'Client wants to implement sustainability-linked incentives and its tracked/verified via wellness app. Action: Send rollout guide and usage analytics.',
 'Client inquired about long-term care insurance and its availability as a voluntary benefit. Action: Send plan designs and communications kit.',
 'Client inquired about financial wellness support and its customizability. Action: Share case study, white-labeling details, and onboarding plan.',
 'Client inquired about global travel insurance with regional restrictions and trip-based activatio

In [76]:
score = bert_scorer.compute(
    predictions=predicted_summaries,
    references=test_summaries,
    lang='en',
    rescale_with_baseline=True
)


In [77]:
sum(score['f1'])/len(score['f1'])

0.5331512361764907

**Much better result**
